In [1]:
import re
import json
from collections import defaultdict
from pathlib import Path
from yarn_utils import YARNGraph
from grewpy import Graph, GRS
import itertools
import networkx as nx

connected to port: 41889


# Load Data

In [5]:
# FOLDER_PATH = "annotations/FRACAS_12032026/"
# FILE = "105h.yarn.json"

FOLDER_PATH = "annotations/"
FILE = "1.yarn.json"

# FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
# FILE = "106h.yarn.json"

# FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
# FILE = "46p.yarn.json"

In [6]:
with open(FOLDER_PATH + FILE) as f:
    yarn_json = json.load(f)

yarn_graph = YARNGraph(yarn_json)
yarn_grew = yarn_graph.grew()

In [61]:
with open('output.json', 'w') as f:
    json.dump(yarn_grew, f)

# Preprocessing

In [63]:
# reifies domain, mod, and poss when they have incoming H edges
# TO DO: make temp an L edge always

grs_path = "grs/main.grs"
grs = GRS(grs_path)
yarn_grew = grs.apply(Graph(yarn_grew), strat='main').json_data()

In [64]:
# save as json
# with open('output.json', 'w') as f:
#     json.dump(yarn_grew, f)

In [95]:
def get_S_descendants(yarn_grew):
    nodes = yarn_grew["nodes"]
    edges = yarn_grew["edges"]

    adj = {}
    for e in edges:
        adj.setdefault(e["src"], []).append(e["tar"])

    result = {}

    for node_id, node_data in nodes.items():
        if node_data.get("type") != "S":
            continue

        visited = set()
        stack = [node_id]
        reachable_V = set()

        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)

            current_type = nodes[current].get("type")

            if current_type in ["L", "H", "V"]:
                reachable_V.add(current)

            if current_type == "C": # or D...
                continue

            for neighbor in adj.get(current, []):
                if neighbor not in visited:
                    stack.append(neighbor)

        result[node_id] = list(reachable_V)

    return result

In [ ]:
def propagate_s_node_information(yarn_grew, s_descendants):

    for s_node, descendants in s_descendants.items():
        for descendant in descendants:
            for node, feats in yarn_grew['nodes'].items():
                if node == descendant:
                    feats['event'] = s_node
    
    return yarn_grew

In [97]:
s_descendants = get_S_descendants(yarn_grew)
yarn_grew = propagate_s_node_information(yarn_grew, s_descendants)

# Build F and R

In [68]:
id2var = {}
def build_F(yarn_grew_graph, id2var, variables):

    def fresh_variable(base='e'):
        i = 0
        while True:
            variable = base if i == 0 else f"{base}{i}"
            if variable not in variables:
                variables.add(variable)
                break
            i += 1
        return variable

    F = []
    nodes = yarn_grew_graph['nodes']
    edges = yarn_grew_graph['edges']

    src_to_tars = defaultdict(list)
    tar_to_srcs = defaultdict(list)
    for edge in edges:
        src_to_tars[edge['src']].append(edge['tar'])
        tar_to_srcs[edge['tar']].append(edge['src'])

    for node, feats in nodes.items():

        if feats['type'] == 'S':
            id2var[node] = feats['var']
            variables.add(feats['var'])

            scope = None
            for src in tar_to_srcs[node]:
                if nodes[src]['type'] == 'C':
                    scope = tar_to_srcs[src][0] if tar_to_srcs[src] else None

            F.append({
                'id': node,
                'scope': scope,
                'incoming': None,
                'outgoing': None,
                'S': feats['event'],
                'type': '∃',
                'variable': id2var[node],
                'tar_label': 'S',
            })

        if (feats['type'] in ['L', 'H']) and feats['feat'] in ['quant', 'temp']:
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    edge_label = feats['value'] if feats['value'] else feats['feat']
                    src = next((s for s in tar_to_srcs[node] if nodes[s]['type'] in ['L', 'H'] and nodes[s]['feat'] == 'quant'), None)

                    if feats['feat'] == 'quant':
                        if edge_label == 'exists':
                            q_type = '∃'
                        elif edge_label == 'forall':
                            q_type = '∀'
                        else:
                            q_type = f'Q_{edge_label}'
                    else:
                        q_type = f'T_{edge_label}'

                    tar_label = nodes[tar].get('pred', nodes[tar].get('concept'))

                    if tar not in id2var:
                        id2var[tar] = fresh_variable(base=tar_label[0])
                    else:
                        raise AssertionError(f"Double quantification. Variable for {tar} already exists in id2var.")

                    F.append({
                        'id': tar,
                        'scope': src if feats['type'] == "H" else None,
                        'incoming': node, # node
                        'outgoing': None,
                        'S': feats['event'],
                        'type': q_type,
                        'variable': id2var[tar],
                        'tar_label': tar_label,
                    })

        if (feats['type'] in ['L', 'H']) and feats['feat'] in ['neg', 'modal', 'aspect']:
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] in ['V', 'L', 'H']:
                    edge_label = feats['value'] if feats['value'] else feats['feat']
                    src = tar_to_srcs[node][0] if tar_to_srcs[node] else None

                    F.append({
                        'id': node,
                        'scope': src,
                        'incoming': None,
                        'outgoing': tar,
                        'S': feats['event'],
                        'type': edge_label,
                        'variable': None,
                        'tar_label': None,
                    })

    for node, feats in nodes.items():
        if feats['type'] == 'V' and node not in id2var:
            id2var[node] = feats.get('pred', feats.get('concept', '')).upper()

    F = [f for f in F if f['type'] not in ['perfective', 'state', 'habitual']]

    return F

In [69]:
F= build_F(yarn_grew, id2var=id2var, variables=set())
F

[{'id': 's1',
  'scope': None,
  'incoming': None,
  'outgoing': None,
  'S': 's1',
  'type': '∃',
  'variable': 's1',
  'tar_label': 'S'},
 {'id': 's2',
  'scope': 'vw1',
  'incoming': None,
  'outgoing': None,
  'S': 's1',
  'type': '∃',
  'variable': 's2',
  'tar_label': 'S'},
 {'id': 'vw1',
  'scope': None,
  'incoming': 'l1',
  'outgoing': None,
  'S': 's1',
  'type': 'T_present',
  'variable': 'w',
  'tar_label': 'want-01'},
 {'id': 'vb1',
  'scope': None,
  'incoming': 'l2',
  'outgoing': None,
  'S': 's1',
  'type': '∀',
  'variable': 'b',
  'tar_label': 'boy'},
 {'id': 'vb2',
  'scope': None,
  'incoming': 'l6',
  'outgoing': None,
  'S': 's2',
  'type': 'T_present',
  'variable': 'b1',
  'tar_label': 'bark-01'},
 {'id': 'vd1',
  'scope': None,
  'incoming': 'l7',
  'outgoing': None,
  'S': 's2',
  'type': '∃',
  'variable': 'd',
  'tar_label': 'dog'},
 {'id': 'l10',
  'scope': 's1-neg',
  'incoming': None,
  'outgoing': 'vw1',
  'S': 's1',
  'type': 'neg',
  'variable': None,

In [ ]:
def add_to_R(R, key, connective, relations):
    if key not in R:
        R[key] = {'and': [], 'or': []}
    R[key][connective].extend(relations)

def build_R(yarn_grew, id2var):

    nodes = yarn_grew['nodes']
    edges = yarn_grew['edges']

    # pre-index edges
    src_to_tars = defaultdict(list)
    tar_to_srcs = defaultdict(list)
    for edge in edges:
        src_to_tars[edge['src']].append(edge['tar'])
        tar_to_srcs[edge['tar']].append(edge['src'])
    
    R = {}
    for node, feats in nodes.items():
        if feats['type'] == "E":
            if feats['rel'].startswith('op'):
                continue

            edge_label = feats['rel']
            for src in tar_to_srcs[node]:
                for tar in src_to_tars[node]:
                    key = tar if id2var[src].isupper() else src
                    if nodes[tar]['concept'] == 'or':
                        grandchildren = [src_to_tars[e][0] for e in src_to_tars[tar]]
                        add_to_R(R, key, 'or', [(edge_label, src, t) for t in grandchildren])
                    elif nodes[tar]['concept'] == 'and':
                        grandchildren = [src_to_tars[e][0] for e in src_to_tars[tar]]
                        add_to_R(R, key, 'and', [(edge_label, src, t) for t in grandchildren])
                    else:
                        add_to_R(R, key, 'and', [(edge_label, src, tar)])

        if feats['type'] == "L" and feats['feat'] == 'num' and feats['value'] == 'plural':
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    add_to_R(R, tar, 'and', [('plural', tar)])

        if feats['type'] == "L" and feats['feat'] == 'def' and feats['value'] == 'definite':
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    add_to_R(R, tar, 'and', [('C', tar)])

        if feats['type'] == "C":
            edge_label = feats['rel']
            for src in tar_to_srcs[node]:
                for tar in src_to_tars[node]:
                    add_to_R(R, tar, 'and', [(edge_label, src, tar)])

        if feats['type'] == "L" and feats['feat'] in ['manner', 'loc', 'dir', 'duration', 'mod', 'freq']:
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    edge_label = feats['value'] if feats['value'] else feats['feat']
                    for src in tar_to_srcs[node]:
                        src = src.split('-')[0]
                        key = src if id2var[tar].isupper() else tar
                        add_to_R(R, key, 'and', [(edge_label, src, tar)])
    return R

In [126]:
yarn_grew

{'nodes': {'vw1': {'pred': 'want-01',
   'type': 'V',
   'var': 'vw1',
   'event': 's1'},
  'vb1': {'concept': 'boy', 'type': 'V', 'var': 'vb1', 'event': 's1'},
  'vj1': {'concept': 'John', 'type': 'V', 'var': 'vj1', 'event': 's1'},
  'vy1': {'concept': 'young', 'type': 'V', 'var': 'vy1', 'event': 's1'},
  'vd1': {'concept': 'dog', 'type': 'V', 'var': 'vd1', 'event': 's2'},
  'vl1': {'pred': 'large-01', 'type': 'V', 'var': 'vl1'},
  'vd2': {'concept': 'definitely', 'type': 'V', 'var': 'vd2', 'event': 's1'},
  'vb2': {'pred': 'bark-01', 'type': 'V', 'var': 'vb2', 'event': 's2'},
  's1': {'event': 's1', 'type': 'S', 'var': 's1'},
  's1-temp': {'feat': 'temp', 'type': 'F', 'var': 's1-temp', 'event': 's1'},
  's1-quant': {'feat': 'quant', 'type': 'F', 'var': 's1-quant', 'event': 's1'},
  's1-num': {'feat': 'num', 'type': 'F', 'var': 's1-num', 'event': 's1'},
  's1-def': {'feat': 'def', 'type': 'F', 'var': 's1-def', 'event': 's1'},
  's1-manner': {'feat': 'manner',
   'type': 'F',
   'var':

In [127]:
R = build_R(yarn_grew, id2var)
R

{'vw1': {'and': [('ARG0', 'vw1', 'vb1')], 'or': []},
 'vb1': {'and': [('mod', 'vb1', 'vy1'), ('name', 'vb1', 'vj1')], 'or': []},
 'vb2': {'and': [('ARG0', 'vb2', 'vd1')], 'or': []},
 'vd1': {'and': [('ARG1', 'vl1', 'vd1')], 'or': []},
 's1': {'and': [('manner', 's1', 'vd2')], 'or': []},
 's2': {'and': [('ARG1', 'vw1', 's2')], 'or': []}}

# Create the Forest

In [189]:
def build_scope_forest(F, R):

    forest = {'nodes':{}, 'edges':[]}

    for i, f in enumerate(F):
        forest['nodes'][i] = {
            'id': f['id'],
            'scope': f['scope'],
            'incoming': f['incoming'],
            'outgoing': f['outgoing'],
            'S': f['S'],
            'type': f['type'],
            'variable': f['variable'],
            'tar_label': f['tar_label'],
            'relations': R.get(f['id'], {'and': [], 'or': []}),
        }

    for k1, v1 in forest['nodes'].items():
        for k2, v2 in forest['nodes'].items():
            if v1['incoming'] and v2['scope'] and v1['incoming'] == v2['scope']:
                forest['edges'].append({'src':k1, 'tar':k2})
            if v1['id'] and v2['outgoing'] and v1['id'] == v2['outgoing']:
                forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})
            if v1['outgoing'] and v2['incoming'] and v1['outgoing'] == v2['incoming']:
                forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

    return forest

In [190]:
forest = build_scope_forest(F, R)
forest

{'nodes': {0: {'id': 's1',
   'scope': None,
   'incoming': None,
   'outgoing': None,
   'S': 's1',
   'type': '∃',
   'variable': 's1',
   'tar_label': 'S',
   'relations': {'and': [('manner', 's1', 'vd2')], 'or': []}},
  1: {'id': 's2',
   'scope': 'vw1',
   'incoming': None,
   'outgoing': None,
   'S': 's1',
   'type': '∃',
   'variable': 's2',
   'tar_label': 'S',
   'relations': {'and': [('ARG1', 'vw1', 's2')], 'or': []}},
  2: {'id': 'vw1',
   'scope': None,
   'incoming': 'l1',
   'outgoing': None,
   'S': 's1',
   'type': 'T_present',
   'variable': 'w',
   'tar_label': 'want-01',
   'relations': {'and': [('ARG0', 'vw1', 'vb1')], 'or': []}},
  3: {'id': 'vb1',
   'scope': None,
   'incoming': 'l2',
   'outgoing': None,
   'S': 's1',
   'type': '∀',
   'variable': 'b',
   'tar_label': 'boy',
   'relations': {'and': [('mod', 'vb1', 'vy1'), ('name', 'vb1', 'vj1')],
    'or': []}},
  4: {'id': 'vb2',
   'scope': None,
   'incoming': 'l6',
   'outgoing': None,
   'S': 's2',
   'ty

## Add the Participant before Event constraint

In [192]:
# Predicates are introduced after their arguments (E relations)
# Arguments are introduced after their predicates (C relations)

def add_participants_before_event_principle(forest, yarn_grew, R): # maybe change the name?

    for _, rels in R.items():
        for rel in rels['and'] + rels['or']:
            if len(rel) == 3:
                src = rel[1]
                tar = rel[2]

                for k1, v1 in forest['nodes'].items():
                    for k2, v2 in forest['nodes'].items():
                        if v1['id'] == src and v2['id'] == tar and \
                            yarn_grew['nodes'][src]['type'] == 'V' and \
                            yarn_grew['nodes'][tar]['type'] == 'V':
                            
                            forest['edges'].append({'src':k2, 'tar':k1})

                        if v1['id'] == src and v2['id'] == tar and \
                            yarn_grew['nodes'][src]['type'] == 'V' and \
                            yarn_grew['nodes'][tar]['type'] == 'S':
                            
                            forest['edges'].append({'src':k1, 'tar':k2}) # participant after event in the case of C edges
    
    return forest

In [193]:
def add_s_node_scope(forest, s_descendants):
    for k1, v1 in forest['nodes'].items():
        if v1['id'] in s_descendants:
            for k2, v2 in forest['nodes'].items():
                if v2['id'] in s_descendants[v1['id']]:
                    forest['edges'].append({'src':k1, 'tar':k2})
                    
    return forest

# Get All Possible Trees

In [2]:
def get_all_possible_trees(n):
    nodes = list(range(n))
    for seq in itertools.product(nodes, repeat=n-2):
        yield nx.from_prufer_sequence(seq)

In [3]:
get_all_possible_trees(8)

<generator object get_all_possible_trees at 0xf0bd10291e00>

In [3]:
def get_all_possible_rooted_directed_trees(nodes): # no need for 'rooted'

    #nodes = list(forest['nodes'].keys())
    n_nodes = len(nodes)

    all_possible_rooted_directed_tree_edges = []

    for tree in get_all_possible_trees(n_nodes):

        for root in nodes:

            visited = set([root])
            stack = [root]
            directed_edges = []

            while stack:
                current = stack.pop()

                for neighbor in tree.neighbors(current):
                    if neighbor not in visited:
                        visited.add(neighbor)
                        stack.append(neighbor)

                        directed_edges.append({'src': current,'tar': neighbor})

            all_possible_rooted_directed_tree_edges.append({'edges': directed_edges})
    
    return all_possible_rooted_directed_tree_edges

In [7]:
get_all_possible_rooted_directed_trees([0, 1, 2, 3, 4, 5, 6, 7])

[{'edges': [{'src': 0, 'tar': 1},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 4},
   {'src': 0, 'tar': 5},
   {'src': 0, 'tar': 6},
   {'src': 0, 'tar': 7}]},
 {'edges': [{'src': 1, 'tar': 0},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 4},
   {'src': 0, 'tar': 5},
   {'src': 0, 'tar': 6},
   {'src': 0, 'tar': 7}]},
 {'edges': [{'src': 2, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 4},
   {'src': 0, 'tar': 5},
   {'src': 0, 'tar': 6},
   {'src': 0, 'tar': 7}]},
 {'edges': [{'src': 3, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 4},
   {'src': 0, 'tar': 5},
   {'src': 0, 'tar': 6},
   {'src': 0, 'tar': 7}]},
 {'edges': [{'src': 4, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 5},
   {'src': 0, 'tar': 6},
   {'src': 0, 'tar': 7}]},
 {'edges': [{'src': 5, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 

# Build T_all

In [196]:
# Gets the children of nodes that don't introduce variables
def get_H_children(graph):
    H_children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        if not graph['nodes'][src]['variable']:
            H_children_dict[src] = H_children_dict.get(src, []) + [tar]

    return H_children_dict

In [197]:
# Get all children
def get_children(graph):
    children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        children_dict[src] = children_dict.get(src, []) + [tar]
    return children_dict

# Extend children to descendants
def get_descendants(node, children_dict, visited=None):
    if visited is None:
        visited = set()
    descendants = []
    for child in children_dict.get(node, []):
        if child not in visited:
            visited.add(child)
            descendants.append(child)
            descendants.extend(get_descendants(child, children_dict, visited))
    return descendants

def get_all_descendants(graph):
    children_dict = get_children(graph)
    descendants_dict = {}
    for node in children_dict:
        descendants_dict[node] = get_descendants(node, children_dict)

    return descendants_dict

In [198]:
# Constraint Checkers

def check_compatibility_of_scopes(tree, forest):
    descendants_tree = get_all_descendants(tree)
    descendants_forest = get_all_descendants(forest)

    for k, v in descendants_forest.items():
        if not v:
            continue
        if k not in descendants_tree:
            return False
        for descendant in v:
            if descendant not in descendants_tree[k]:
                return False
    return True

def check_locality_of_features(tree, forest):
    children_tree = get_children(tree)
    H_children_forest = get_H_children(forest)
    
    for k, v in H_children_forest.items():
        if not v:
            continue
        if k not in children_tree:
            return False
        for child in v:
            if child not in children_tree[k]:
                return False
    return True

In [199]:
def build_T_all(forest, valid_tree_edges):

    T_all = []
    for tree in valid_tree_edges:
        new_tree = forest.copy()
        new_tree['edges'] = tree['edges']
        T_all.append(new_tree)
    
    return T_all

## Reformat T_all

In [200]:
def reformat(graph):
    all_targets = {edge["tar"] for edge in graph["edges"]}
    root_id = next(nid for nid in graph["nodes"] if nid not in all_targets)
    
    def build(node_id):
        node = dict(graph["nodes"][node_id])
        node["children"] = [build(edge["tar"]) for edge in graph["edges"] if edge["src"] == node_id]
        return node
    
    return build(root_id)

# Interpretation

In [201]:
def fmt_rel(rel, id2var):
    pred = rel[0]
    if len(rel) == 3:
        return f"{pred}({id2var[rel[1]]},{id2var[rel[2]]})"
    else:
        return f"{pred}({id2var[rel[1]]})"

def get_relations(node, id2var):
    and_rels = [fmt_rel(rel, id2var) for rel in node['relations']['and']]
    or_rels  = [fmt_rel(rel, id2var) for rel in node['relations']['or']]
    return and_rels + [f"({' ∨ '.join(or_rels)})"] if or_rels else and_rels

In [202]:
def conj(parts):
    parts = [p for p in parts if p and p.strip()]
    return " ∧ ".join(parts)


def wrap_quant(q, var, head, relations, bodies, connective="∧"):
    rel = conj(relations)
    head_part = f"{head}({var})"
    left = f"{head_part} ∧ {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}{var}. ( {left}\n {connective} ({body}) )"
    else:
        return f"{q}{var}. ( {left} )"
    
def wrap_generalized_quant(q, var, gen_quant, head, relations, bodies, connective="∧"):
    rel = conj(relations)
    head_part = f"{head}({var}) ∧ {gen_quant}({var})"
    left = f"{head_part} ∧ {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}{var}. ( {left}\n {connective} ({body}) )"
    else:
        return f"{q}{var}. ( {left} )"
    
def wrap_temp(q, var, S, head, relations, bodies, connective="∧"):
    rel = conj(relations)
    head_part = f"{head}({var},{S})"
    left = f"{head_part} ∧ {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}{var}. ( {left}\n {connective} ({body}) )"
    else:
        return f"{q}{var}. ( {left} )"

def clean_formula(formula):
    clean_formula = formula.replace(" ∧ ()", "")
    return clean_formula

In [203]:
def interpret(root, temp_variable, id2var):
    
    if root is None:
        return ""

    child_formulas = [interpret(child, temp_variable, id2var) for child in root["children"]]
    relations = get_relations(root, id2var)

    if root["type"] == "∃":
        return wrap_quant("∃", root["variable"], root["tar_label"], relations, child_formulas, connective="∧")
    
    if root["type"] == "∀":
        return wrap_quant("∀", root["variable"], root["tar_label"], relations, child_formulas, connective="→")
    
    if root["type"].startswith("Q_"):
        gen_quant = root["type"][2:] # disallow whitespaces
        return wrap_generalized_quant("∃", root["variable"], gen_quant, root["tar_label"], relations, child_formulas, connective="∧")
    
    if root["type"] == "T_present":
        temp = f"{root['variable']}_O_{temp_variable}"
        child_formulas = [interpret(child, root["variable"], id2var) for child in root["children"]]
        return wrap_temp("∃", root["variable"], root['S'], root["tar_label"], relations + [temp], child_formulas, connective="∧")

    if root["type"] == "T_past":
        temp = f"{root['variable']}≺{temp_variable}"
        child_formulas = [interpret(child, root["variable"], id2var) for child in root["children"]]
        return wrap_temp("∃", root["variable"], root['S'], root["tar_label"], relations + [temp], child_formulas, connective="∧")
    
    if root["type"] == "T_future":
        temp = f"{temp_variable}≺{root['variable']}"
        child_formulas = [interpret(child, root["variable"], id2var) for child in root["children"]]
        return wrap_temp("∃", root["variable"], root['S'], root["tar_label"], relations + [temp], child_formulas, connective="∧")
    
    if root["type"] == "neg":
        return f"¬ ( {conj(child_formulas)} )"
    
    if root["type"] == "possibility":
        return f"◇ ( {conj(child_formulas)} )"

    if root["type"] == "necessity":
        return f"□ ( {conj(child_formulas)} )"

In [204]:
def extract_number(path):

    match = re.match(r"(\d+)", path.name)
    return int(match.group(1)) if match else float("inf")


def load_yarn(input_path, recursive=False):
    if isinstance(input_path, (str, Path)):
        input_path = [input_path]

    all_files = []

    for path in input_path:
        path = Path(path)

        if path.is_file():
            if path.name.endswith(".yarn.json"):
                all_files.append(path)

        elif path.is_dir():
            files = path.rglob("*.yarn.json") if recursive else path.glob("*.yarn.json")
            all_files.extend(files)

        else:
            raise FileNotFoundError(f"{path} does not exist")

    all_files = sorted(all_files, key=extract_number)

    graphs = []
    for file_path in all_files:
        with open(file_path, "r", encoding="utf-8") as f:
            graph = json.load(f)
            if graph.get('labels'):  # safer
                graphs.append((file_path, graph))

    return graphs

In [208]:
import traceback
import multiprocessing as mp


TIMEOUT = 20


def process_one(path, yarn_graph_json, output_queue):
    try:
        id2var = {}
        variables = set()

        yarn_graph = YARNGraph(yarn_graph_json)
        yarn_grew = yarn_graph.grew()
        yarn_grew = grs.apply(Graph(yarn_grew), strat='main').json_data()
        
        s_descendants = get_S_descendants(yarn_grew)
        yarn_grew = propagate_s_node_information(yarn_grew, s_descendants)

        # save as json
        # with open('output.json', 'w') as f:
        #     json.dump(yarn_grew, f)

        F = build_F(yarn_grew, id2var, variables)
        R = build_R(yarn_grew, id2var)
        forest = build_scope_forest(F, R)
        # print(forest['nodes'])
        # print(forest['edges'])
        
        forest = add_participants_before_event_principle(forest, yarn_grew, R)
        # print(forest['edges'])
        forest = add_s_node_scope(forest, s_descendants)
        # print(forest['edges'])
        
        all_possible_rooted_directed_tree_edges = get_all_possible_rooted_directed_trees(forest)

        valid_tree_edges = [
            tree for tree in all_possible_rooted_directed_tree_edges
            if check_locality_of_features(tree, forest)
        ]
        valid_tree_edges = [
            tree for tree in valid_tree_edges
            if check_compatibility_of_scopes(tree, forest)
        ]

        T_all = build_T_all(forest, valid_tree_edges)
        T_all = [reformat(tree) for tree in T_all]

        results = []
        for T in T_all:
            results.append(clean_formula(interpret(T, 'now', id2var)))

        output_queue.put({
            "path": str(path),
            "meta": yarn_graph_json.get("meta", {}),
            "results": results,
            "error": None
        })

    except Exception as e:
        output_queue.put({
            "path": str(path),
            "meta": yarn_graph_json.get("meta", {}),
            "results": None,
            "error": traceback.format_exc()
        })

def yarn2fol(yarn_graphs):
    for path, yarn_graph_json in yarn_graphs:

        print("\nProcessing:", path)

        output_queue = mp.Queue()
        p = mp.Process(target=process_one, args=(path, yarn_graph_json, output_queue))

        p.start()
        p.join(TIMEOUT)

        if p.is_alive():
            p.terminate()
            p.join()

            print(path)
            print("TIMEOUT after", TIMEOUT, "seconds")
            continue

        if output_queue.empty():
            print(path)
            print("No output returned")
            continue

        result = output_queue.get()

        if result["error"]:
            print(path)
            print("ERROR:")
            print(result["error"])
            continue

        print(result["path"])
        print(result["meta"].get("type"), ":", result["meta"].get("text"))

        for r in result["results"]:
            print(r)
            print("---")

In [209]:
FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
FILE = "92h.yarn.json"

corpus = load_yarn(FOLDER_PATH)
yarn2fol(corpus)


Processing: annotations/FRACAS_1premise_yesno/1p.yarn.json


annotations/FRACAS_1premise_yesno/1p.yarn.json
premise : An Italian became the world's greatest tenor.
∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY)
 ∧ (∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃w. ( world(w) ∧ C(w)
 ∧ (∃b. ( become-01(b,s1) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺now )) )) )) )) )
---
∃s1. ( S(s1)
 ∧ (∃w. ( world(w) ∧ C(w) ) ∧ ∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY)
 ∧ (∃b. ( become-01(b,s1) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺now )) )) )) )
---
∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY)
 ∧ (∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃b. ( become-01(b,s1) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺now
 ∧ (∃w. ( world(w) ∧ C(w) )) )) )) )) )
---
∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY)
 ∧ (∃w. ( world(w) ∧ C(w) ) ∧ ∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧ C(t)
 ∧ (∃b. ( become-01(b,s1) ∧ ARG1(b,p) ∧ ARG2(b,t) ∧ b≺now )) )) )) )
---
∃s1. ( S(s1)
 ∧ (∃p. ( person(p) ∧ mod(p,COUNTRY)
 ∧ (∃w. ( world(w) ∧ C(w)
 ∧ (∃t. ( tenor(t) ∧ ARG1(GREAT-02,t) ∧

In [145]:
FOLDER_PATH = "annotations/"
FILE = "1.yarn.json"

corpus = load_yarn(FOLDER_PATH+FILE)
yarn2fol(corpus)


Processing: annotations/1.yarn.json
annotations/1.yarn.json
None : Every young boy called John definitely doesn't want a large dog to bark.
∃s1. ( S(s1) ∧ manner(s1,DEFINITELY)
 ∧ (∀b. ( boy(b) ∧ mod(b,YOUNG) ∧ name(b,JOHN)
 → (¬ ( ∃w. ( want-01(w,s1) ∧ ARG0(w,b) ∧ w_O_NOW
 ∧ (∃s2. ( S(s2) ∧ ARG1(w,s2)
 ∧ (∃d. ( dog(d) ∧ ARG1(LARGE-01,d)
 ∧ (∃b1. ( bark-01(b1,s2) ∧ ARG0(b1,d) ∧ b1_O_w )) )) )) ) )) )) )
---


In [210]:
p = """
[1] ?[S1]: (s(S1)
  & (?[M]: (meeting(M) & loc(S1,M) & c(M)
  & (![P]: (person(P) & location(P,M) & c(P)
  => (?[C]: (chairman(C) & arg1(new-01,C)
  & (?[V]: (vote_01(V,S1) & arg0(V,P) & arg1(V,C) & precedes(V,now))))))))))
"""
h = """
[1] ?[S1]: (s(S1)
  & (?[M]: (meeting(M) & loc(S1,M) & c(M)
  & (![P]: (person(P) & location(P,M) & c(P)
  => (?[C]: (chairman(C) & arg1(new-01,C)
  & (?[V]: (vote_01(V,S1) & arg0(V,P) & arg1(V,C) & precedes(V,now))))))))))
"""
p == h

True

# Vampire

In [182]:
def fmt_rel(rel, id2var):
    pred = rel[0].lower().replace("-", "_")
    
    def fmt_arg(arg):
        val = id2var[arg]
        if val.isupper():  # constant
            return val.lower()
        else:  # variable
            return val.upper()
    
    if len(rel) == 3:
        return f"{pred}({fmt_arg(rel[1])},{fmt_arg(rel[2])})"
    else:
        return f"{pred}({fmt_arg(rel[1])})"

def get_relations(node, id2var):
    and_rels = [fmt_rel(rel, id2var) for rel in node['relations']['and']]
    or_rels  = [fmt_rel(rel, id2var) for rel in node['relations']['or']]
    return and_rels + [f"({' | '.join(or_rels)})"] if or_rels else and_rels

def conj(parts):
    parts = [p for p in parts if p and p.strip()]
    return " & ".join(parts)

def wrap_quant(q, var, head, relations, bodies, connective="&"):
    rel = conj(relations)
    head_part = f"{head.lower().replace('-', '_')}({var.upper()})"
    left = f"{head_part} & {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        if connective == "=>":
            return f"{q}[{var.upper()}]: ({left}\n  => ({body}))"
        else:
            return f"{q}[{var.upper()}]: ({left}\n  & ({body}))"
    else:
        return f"{q}[{var.upper()}]: ({left})"

def wrap_generalized_quant(q, var, gen_quant, head, relations, bodies, connective="&"):
    rel = conj(relations)
    head_part = f"{head.lower().replace('-', '_')}({var.upper()}) & {gen_quant.lower()}({var.upper()})"
    left = f"{head_part} & {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}[{var.upper()}]: ({left}\n  & ({body}))"
    else:
        return f"{q}[{var.upper()}]: ({left})"

def wrap_temp(q, var, S, head, relations, bodies, connective="&"):
    rel = conj(relations)
    head_part = f"{head.lower().replace('-', '_')}({var.upper()},{S.upper()})"
    left = f"{head_part} & {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}[{var.upper()}]: ({left}\n  & ({body}))"
    else:
        return f"{q}[{var.upper()}]: ({left})"

def clean_formula(formula):
    return formula.replace(" & ()", "")

def interpret(root, temp_variable, id2var):

    if root is None:
        return ""

    child_formulas = [interpret(child, temp_variable, id2var) for child in root["children"]]
    relations = get_relations(root, id2var)

    if root["type"] == "∃":
        return wrap_quant("?", root["variable"], root["tar_label"], relations, child_formulas)

    if root["type"] == "∀":
        return wrap_quant("!", root["variable"], root["tar_label"], relations, child_formulas, connective="=>")

    if root["type"].startswith("Q_"):
        gen_quant = root["type"][2:]
        return wrap_generalized_quant("?", root["variable"], gen_quant, root["tar_label"], relations, child_formulas)

    if root["type"] == "T_present":
        temp = f"overlap({root['variable'].upper()},{temp_variable.upper()})"
        child_formulas = [interpret(child, root["variable"], id2var) for child in root["children"]]
        return wrap_temp("?", root["variable"], root['S'], root["tar_label"], relations + [temp], child_formulas)

    if root["type"] == "T_past":
        temp = f"precedes({root['variable'].upper()},{temp_variable.upper()})"
        child_formulas = [interpret(child, root["variable"], id2var) for child in root["children"]]
        return wrap_temp("?", root["variable"], root['S'], root["tar_label"], relations + [temp], child_formulas)

    if root["type"] == "T_future":
        temp = f"precedes({temp_variable.upper()},{root['variable'].upper()})"
        child_formulas = [interpret(child, root["variable"], id2var) for child in root["children"]]
        return wrap_temp("?", root["variable"], root['S'], root["tar_label"], relations + [temp], child_formulas)

    if root["type"] == "neg":
        return f"~({conj(child_formulas)})"

    if root["type"] == "possibility":
        return f"possibly({conj(child_formulas)})"

    if root["type"] == "necessity":
        return f"necessarily({conj(child_formulas)})"

In [ ]:


def generate_valid_trees(forest):
    nodes = list(forest['nodes'].keys())
    forest_descendants = get_all_descendants(forest)
    H_children_forest = get_H_children(forest)

    # Step 1: select valid roots
    # A valid root is any node not in the scope of something else
    scoped_nodes = set()
    for node, descendants in forest_descendants.items():
        for d in descendants:
            scoped_nodes.add(d)
    valid_roots = [n for n in nodes if n not in scoped_nodes]

    all_trees = []

    def get_partial_descendants(node, partial_children):
        descendants = []
        for child in partial_children.get(node, []):
            descendants.append(child)
            descendants.extend(get_partial_descendants(child, partial_children))
        return descendants

    def check_scope(node, new_child, partial_children, partial_parent):
        """
        After adding new_child under node, check compatibility of scopes.
        For every forest ancestor of new_child, new_child must be
        a descendant of that ancestor in the partial tree too.
        """
        # Find forest ancestors of new_child
        forest_ancestors = [k for k, v in forest_descendants.items() if new_child in v]

        for ancestor in forest_ancestors:
            # ancestor must also be an ancestor of new_child in the partial tree
            # i.e. new_child must be reachable from ancestor
            if ancestor not in partial_children and ancestor != node:
                # ancestor hasn't been placed yet — can't verify, skip
                continue
            # Check if ancestor is an ancestor of node in partial tree
            # by walking up via partial_parent
            current = node
            found = (current == ancestor)
            while current in partial_parent and not found:
                current = partial_parent[current]
                found = (current == ancestor)
            if not found:
                return False

        # Also check: for every forest descendant of new_child,
        # if already placed, it must be under new_child
        for required_desc in forest_descendants.get(new_child, []):
            placed_under = None
            for n, children in partial_children.items():
                if required_desc in get_partial_descendants(n, partial_children):
                    placed_under = n
                    break
            if placed_under is not None:
                # must be under new_child
                if required_desc not in get_partial_descendants(new_child, partial_children):
                    return False

        return True

    def build_tree(current_frontier, unplaced, partial_children, partial_parent, edges):
        """
        current_frontier: nodes at the current depth level whose children we are assigning
        unplaced: nodes not yet in the tree
        partial_children: dict node -> [children]
        partial_parent: dict node -> parent
        edges: list of directed edges so far
        """
        if not unplaced:
            all_trees.append({'edges': list(edges)})
            return

        # For each node in the current frontier, assign children from unplaced
        # We do this recursively: pick a frontier node, assign its children,
        # then recurse with the next frontier node
        def assign_children_to_frontier(frontier_idx, unplaced, partial_children, partial_parent, edges, next_frontier):
            if frontier_idx == len(current_frontier):
                # All frontier nodes have been assigned children
                # next_frontier becomes the new frontier
                if not next_frontier and unplaced:
                    # No new frontier but still unplaced nodes — dead end
                    return
                build_tree(next_frontier, unplaced, partial_children, partial_parent, edges)
                return

            parent = current_frontier[frontier_idx]

            # Step 2: add mandatory H-children (locality of features) first
            mandatory = [c for c in H_children_forest.get(parent, []) if c in unplaced]
            
            # Check mandatory children don't violate scope
            valid_mandatory = True
            temp_children = dict(partial_children)
            temp_parent = dict(partial_parent)
            temp_edges = list(edges)
            temp_unplaced = list(unplaced)
            temp_next_frontier = list(next_frontier)

            for child in mandatory:
                if not check_scope(parent, child, temp_children, temp_parent):
                    valid_mandatory = False
                    break
                temp_children[parent] = temp_children.get(parent, []) + [child]
                temp_parent[child] = parent
                temp_edges.append({'src': parent, 'tar': child})
                temp_unplaced.remove(child)
                temp_next_frontier.append(child)

            if not valid_mandatory:
                # Can't satisfy locality — prune this branch
                return

            # Step 3: assign optional children from remaining unplaced
            # (excluding mandatory ones already assigned)
            remaining_unplaced = [n for n in temp_unplaced if n not in H_children_forest.get(parent, [])]

            # Try all subsets of remaining_unplaced as optional children of parent
            for r in range(0, len(remaining_unplaced) + 1):
                for optional_children in itertools.combinations(remaining_unplaced, r):
                    
                    curr_children = dict(temp_children)
                    curr_parent = dict(temp_parent)
                    curr_edges = list(temp_edges)
                    curr_unplaced = list(temp_unplaced)
                    curr_next_frontier = list(temp_next_frontier)
                    valid = True

                    for child in optional_children:
                        # Step 3: check scope before adding
                        if not check_scope(parent, child, curr_children, curr_parent):
                            valid = False
                            break
                        curr_children[parent] = curr_children.get(parent, []) + [child]
                        curr_parent[child] = parent
                        curr_edges.append({'src': parent, 'tar': child})
                        curr_unplaced.remove(child)
                        curr_next_frontier.append(child)

                    if not valid:
                        continue  # Step 4: skip this combination

                    # Move to next frontier node
                    assign_children_to_frontier(
                        frontier_idx + 1,
                        curr_unplaced,
                        curr_children,
                        curr_parent,
                        curr_edges,
                        curr_next_frontier
                    )

        assign_children_to_frontier(0, unplaced, partial_children, partial_parent, edges, [])

    # Try each valid root
    for root in valid_roots:
        unplaced = [n for n in nodes if n != root]
        build_tree(
            current_frontier=[root],
            unplaced=unplaced,
            partial_children={},
            partial_parent={},
            edges=[]
        )

    return all_trees